# Stage 2 BindCraft smoke — restart-safe thin front end (cells A–H)

Reliability-consolidation version. All logic lives in pinned project scripts
(`scripts/stage2_*.py`); this notebook only provisions and launches them.

- **Frozen science**: EGFR Domain III 310–481; envelope B 390–403 + 421–431;
  hotspots `390,393,399,421,424,431`; binder 80 aa; cap 3 trajectories;
  PDL1 control hotspot 56 / 65 aa / cap 1; default filters + official
  4stage multimer advanced settings (only `max_trajectories` changed).
- Cell D also applies the one auditable, pinned upstream patch (D-019): a failed per-model PyRosetta relaxation is recorded, the unrelaxed PDB is retained, and the run continues instead of dying.
- Run top to bottom. After a Colab runtime reset, re-run **A→G**: completed
  jobs are checkpointed on Drive and skipped automatically (PDL1 ~30 min is
  never needlessly re-run).
- GPU quota denial is a controlled stop (`COMPUTE_BLOCKED`); there is **no
  CPU fallback**. No production sampling is performed by this notebook.


In [ ]:
# Cell A — GPU gate + hardware metadata. Runs first; never falls back to CPU.
import json, os, platform, shutil, subprocess, sys

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def gpu_query(field):
    return sh(f"nvidia-smi --query-gpu={field} --format=csv,noheader,nounits 2>/dev/null")

gpu_name = gpu_query('name')
env_meta = {
    'colab_system_python': sys.version.split()[0],
    'platform': platform.platform(),
    'gpu_name': gpu_name or None,
    'gpu_vram_total_mib': int(x) if (x := gpu_query('memory.total')) else None,
    'gpu_vram_free_mib': int(x) if (x := gpu_query('memory.free')) else None,
    'host_ram_total_gb': round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1),
    'disk_free_gb_content': round(shutil.disk_usage('/content').free / 1e9, 1),
    'nvidia_driver': sh('nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null') or None,
}
print(json.dumps(env_meta, indent=2))
if not gpu_name:
    print('GPU_UNAVAILABLE: no NVIDIA GPU is visible (runtime reset or non-GPU backend).')
    print('COMPUTE_QUOTA_BLOCKED: if Colab says "Cannot connect to GPU backend due to usage limits.", this is a quota condition.')
    print('Do NOT continue on CPU. Reconnect a GPU runtime and re-run cells A-G; completed work is checkpointed on Drive and will be skipped.')
    raise SystemExit(2)
print('GPU AVAILABLE:', gpu_name)


In [ ]:
# Cell B — mount persistent Drive; every expensive artifact lives under this root.
import os
from google.colab import drive
drive.mount('/content/drive')
PERSISTENT_ROOT = '/content/drive/MyDrive/BindCraft/stage2_smoke'
os.environ['STAGE2_PERSISTENT_ROOT'] = PERSISTENT_ROOT
os.makedirs(PERSISTENT_ROOT, exist_ok=True)
print('persistent root ->', PERSISTENT_ROOT)


In [ ]:
# Cell C — fetch THIS project at an exact pinned commit (private repo).
# Token comes from Colab Secrets (secret name: GITHUB_TOKEN); it is passed via
# a git header, never printed/logged. Manual tarball upload remains a fallback.
import hashlib, json, os, subprocess, tarfile
from pathlib import Path

REPO_DIR = '/content/egfr-binder-challenge'
PROJECT_PIN = '452ac95c810614ad5c7602ef652644a4c63382c6'
REPO_URL_PUBLIC = 'https://github.com/SyzenShen/egfr-binder-challenge.git'

def runq(argv, **kw):
    return subprocess.run(argv, capture_output=True, text=True, **kw)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    print('no Colab secret GITHUB_TOKEN:', type(exc).__name__)

project_commit = None
if os.getenv('SKIP_GIT_FETCH') and os.path.isdir(REPO_DIR):
    print('SKIP_GIT_FETCH set; using existing', REPO_DIR)
elif os.path.isdir(os.path.join(REPO_DIR, '.git')):
    runq(['git', '-C', REPO_DIR, 'fetch', '--quiet', 'origin', 'main'])
    runq(['git', '-C', REPO_DIR, 'checkout', '--quiet', PROJECT_PIN])
else:
    cmd = ['git']
    if token:
        cmd += ['-c', f'http.extraheader=AUTHORIZATION: bearer {token}']
    cmd += ['clone', '--quiet', REPO_URL_PUBLIC, REPO_DIR]
    p = runq(cmd)
    if p.returncode != 0:
        print('git clone failed (private repo: add Colab secret GITHUB_TOKEN, or use manual upload below).')
        print(p.stderr[-1500:])
    else:
        runq(['git', '-C', REPO_DIR, 'checkout', '--quiet', PROJECT_PIN])

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    project_commit = runq(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).stdout.strip()
    assert project_commit == PROJECT_PIN, f'pin mismatch: {project_commit} != {PROJECT_PIN}'
    print('PROJECT PIN:', project_commit)
elif not os.path.isdir(REPO_DIR):
    print('Manual fallback: upload the repo snapshot tarball (it must contain egfr-binder-challenge/).')
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        if name.endswith(('.tar.gz', '.tgz', '.tar')):
            arc = '/content/' + name
            open(arc, 'wb').write(data)
            with tarfile.open(arc, 'r:gz' if name.endswith('.gz') else 'r') as tf:
                tf.extractall('/content/')
            print('extracted', name)

required = [
    'scripts/stage2_paths.py', 'scripts/stage2_preflight.py',
    'scripts/ensure_af2_weights.py', 'scripts/stage2_configure.py',
    'scripts/stage2_checkpoint.py', 'scripts/stage2_run_job.py',
    'scripts/stage2_analyze.py', 'scripts/stage2_orchestrate.py',
    'scripts/analyze_bindcraft_run.py',
    'data/processed/6ARU_chainA_domain3_310-481.pdb',
]
missing = [r for r in required if not os.path.isfile(os.path.join(REPO_DIR, r))]
assert not missing, f'missing project artifacts: {missing}'

artifacts = {
    'project_commit': project_commit,
    'project_commit_verified': project_commit == PROJECT_PIN,
    'crop_pdb_sha256': sha256_file(os.path.join(REPO_DIR, required[-1])),
    'script_sha256': {r: sha256_file(os.path.join(REPO_DIR, r)) for r in required[:-1]},
}
meta = Path(PERSISTENT_ROOT) / 'persistent' / 'metadata'
meta.mkdir(parents=True, exist_ok=True)
(meta / 'artifacts.json').write_text(json.dumps(artifacts, indent=2))
print('artifacts recorded; crop PDB sha256 =', artifacts['crop_pdb_sha256'])


In [ ]:
# Cell D — BindCraft pinned checkout + isolated py3.10/jax0.6.0 env.
# Idempotent: after a runtime reset this rebuilds /content from scratch.
import os, json, shutil, subprocess

ENV_PREFIX = '/content/bindcraft_env'
MF_PREFIX = '/content/miniforge3'
MAMBA = f'{MF_PREFIX}/bin/mamba'
BINDPY = f'{ENV_PREFIX}/bin/python'
BINDCRAFT_DIR = '/content/bindcraft'
EGFR_DIR = '/content/egfr'
PINNED = '7713aa0d0d351e4117a8befeb8541f3a8ebd3368'
COLABDESIGN_PIN = 'e31a56fe1d9b4de25c8697f3a28b75892941cc72'
CUDA_OVERRIDE = '12.6'  # conda-forge jaxlib 0.6.0 cuda build needs cuda-version >=12.6,<13

def runv(argv, env=None, quiet=False):
    p = subprocess.run(argv, capture_output=True, text=True, env=env)
    if p.returncode != 0:
        print('FAILED:', ' '.join(str(a) for a in argv[:4]), '...')
        print(p.stdout[-4000:]); print(p.stderr[-4000:])
        raise SystemExit(p.returncode)
    if not quiet:
        print('ok:', ' '.join(str(a) for a in argv[:4]))
    return p

if not os.path.isfile(f'{BINDCRAFT_DIR}/.pinned_sha'):
    print('cloning BindCraft pinned commit')
    runv(['git', 'clone', '--quiet', 'https://github.com/martinpacesa/BindCraft', BINDCRAFT_DIR])
    runv(['git', '-C', BINDCRAFT_DIR, 'checkout', '--quiet', PINNED])
    os.chmod(f'{BINDCRAFT_DIR}/functions/dssp', 0o755)
    os.chmod(f'{BINDCRAFT_DIR}/functions/DAlphaBall.gcc', 0o755)
    open(f'{BINDCRAFT_DIR}/.pinned_sha', 'w').write(PINNED)
sha = runv(['git', '-C', BINDCRAFT_DIR, 'rev-parse', 'HEAD'], quiet=True).stdout.strip()
assert sha == PINNED, f'unexpected checkout {sha}'

# [D-019] Apply the pinned, auditable per-model PyRosetta relax-tolerance
# patch BEFORE the long env build. Idempotent (ALREADY_APPLIED on rerun after
# reset); fails loudly on commit mismatch or a partially-patched tree.
# The patch makes a failed per-model relaxation print [STAGE2_RELAX_FAILURE],
# append a record to relax_failures.jsonl, retain the unrelaxed PDB,
# and continue to the next candidate instead of killing the whole run.
# Stdlib-only applier -> system python3 works before the env exists.
patch_meta = f'{PERSISTENT_ROOT}/persistent/metadata/bindcraft_patch.json'
pp = subprocess.run(['python3', f'{REPO_DIR}/scripts/apply_bindcraft_patch.py',
                     '--bindcraft-dir', BINDCRAFT_DIR,
                     '--patch', f'{REPO_DIR}/patches/bindcraft-7713aa0-relax-tolerance.patch',
                     '--out', patch_meta], capture_output=True, text=True)
print(pp.stdout[-2000:])
assert pp.returncode == 0, ('BindCraft relax-tolerance patch failed; '
                            'do not continue\n' + pp.stderr[-2000:])
_pm = json.load(open(patch_meta))
assert _pm['status'] in ('APPLIED', 'ALREADY_APPLIED'), _pm
print('relax-tolerance patch:', _pm['status'])

if not os.path.exists(f'{ENV_PREFIX}/.env_ready'):
    if not os.path.exists(MAMBA):
        print('installing Miniforge (isolated; Colab system Python stays untouched)')
        runv(['wget', '-q',
              'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh',
              '-O', '/tmp/miniforge.sh'])
        runv(['bash', '/tmp/miniforge.sh', '-b', '-p', MF_PREFIX])
    spec = ['python=3.10', 'pip', 'pandas', 'matplotlib', 'numpy<2.0.0',
            'biopython', 'scipy', 'pdbfixer', 'seaborn', 'libgfortran5',
            'tqdm', 'jupyter', 'ffmpeg', 'fsspec', 'py3dmol', 'chex',
            'dm-haiku', 'flax<0.10.0', 'dm-tree', 'joblib', 'ml-collections',
            'immutabledict', 'optax', 'jax=0.6.0', 'jaxlib=0.6.0=*cuda*',
            'cuda-nvcc', 'cudnn']
    env = dict(os.environ, CONDA_OVERRIDE_CUDA=CUDA_OVERRIDE)
    print('creating conda env from upstream spec (this is the long step)...')
    runv([MAMBA, 'create', '-y', '-p', ENV_PREFIX,
          '-c', 'conda-forge', '-c', 'nvidia', *spec], env=env)
    print('installing ColabDesign pinned commit with --no-deps (upstream pattern)')
    runv([f'{ENV_PREFIX}/bin/pip', 'install', '--no-deps',
          f'git+https://github.com/sokrypton/ColabDesign.git@{COLABDESIGN_PIN}'])
    print('installing PyRosetta wheel (~1.6 GB; academic/non-commercial license)')
    runv([f'{ENV_PREFIX}/bin/pip', 'install', 'pyrosetta', '--find-links',
          'https://west.rosettacommons.org/pyrosetta/quarterly/release.cxx11thread.serialization'])
    runv([BINDPY, '-c', 'import sys; assert sys.version_info[:2] == (3, 10)'], quiet=True)
    open(f'{ENV_PREFIX}/.env_ready', 'w').write('ok\n')
    print('environment ready')
else:
    print('reusing existing', ENV_PREFIX)

os.makedirs(EGFR_DIR, exist_ok=True)
shutil.copy2(f'{REPO_DIR}/data/processed/6ARU_chainA_domain3_310-481.pdb',
             f'{EGFR_DIR}/6ARU_chainA_domain3_310-481.pdb')
res = {}
for line in open(f'{EGFR_DIR}/6ARU_chainA_domain3_310-481.pdb'):
    if line.startswith('ATOM') and line[12:16].strip() == 'CA':
        res[int(line[22:26])] = line[17:20].strip()
aa3 = {'Q': 'GLN', 'E': 'GLU', 'D': 'ASP', 'K': 'LYS', 'R': 'ARG', 'N': 'ASN', 'T': 'THR'}
expect = {390: 'Q', 393: 'D', 399: 'K', 421: 'E', 424: 'E', 431: 'K'}
assert sorted(res) == list(range(310, 482)), f'crop numbering wrong: {len(res)} residues'
assert all(res[p] == aa3[a] for p, a in expect.items()), 'hotspot identity mismatch'
print('BindCraft at', sha, '|', runv([BINDPY, '-V'], quiet=True).stdout.strip(),
      '| crop verified:', len(res), 'residues')


In [ ]:
# Cell E — preflight gate inside the isolated env (before any weight download).
import json, os, subprocess
job_env = dict(os.environ)
job_env['LD_LIBRARY_PATH'] = f'{ENV_PREFIX}/lib:' + job_env.get('LD_LIBRARY_PATH', '')
job_env['PATH'] = f'{ENV_PREFIX}/bin:' + job_env.get('PATH', '')
pf_path = f'{PERSISTENT_ROOT}/persistent/metadata/preflight.json'
pfp = subprocess.run([BINDPY, f'{REPO_DIR}/scripts/stage2_preflight.py', '--out', pf_path],
                     capture_output=True, text=True, env=job_env)
print(pfp.stdout)
if pfp.stderr.strip():
    print(pfp.stderr[-2000:])
assert pfp.returncode == 0, 'PRE-FLIGHT FAILED -> environment failure; stop here'
pf = json.load(open(pf_path))
assert pf['passed'] and pf['gpu']['platform'] == 'gpu'
assert pf['checks'] and all(c['ok'] for c in pf['checks'])
print('PRE-FLIGHT PASSED on', pf['gpu']['device_kind'],
      '| jax', pf['versions']['jax'])


In [ ]:
# Cell F — AF2 weights: local exact-set -> Drive cache -> observed download.
# Authority is the exact 15 official .npz files; done.txt is never trusted.
wf_path = f'{PERSISTENT_ROOT}/persistent/metadata/af2_weights.json'
wfp = subprocess.run([
    BINDPY, f'{REPO_DIR}/scripts/ensure_af2_weights.py',
    '--params-dir', f'{BINDCRAFT_DIR}/params',
    '--cache-dir', f'{PERSISTENT_ROOT}/persistent/cache/alphafold',
    '--log-dir', f'{PERSISTENT_ROOT}/persistent/logs',
    '--out', wf_path], capture_output=True, text=True, env=job_env)
print(wfp.stdout[-4000:])
if wfp.stderr.strip():
    print(wfp.stderr[-2000:])
assert wfp.returncode == 0, 'AF2 weight provisioning failed; see persistent/logs'
weights = json.load(open(wf_path))
assert weights['ok'] and weights['validation']['present_count'] == 15
print('AF2 WEIGHTS READY:', weights['action'])


In [ ]:
# Cell G — RUN OR RESUME STAGE 2 SMOKE (one resumable command).
# Re-running after a reset skips valid checkpoints (PDL1 ~30 min included).
g = subprocess.run([
    BINDPY, f'{REPO_DIR}/scripts/stage2_orchestrate.py',
    '--repo-dir', REPO_DIR,
    '--bindcraft-dir', BINDCRAFT_DIR,
    '--bindpy', BINDPY,
    '--root', PERSISTENT_ROOT],
    capture_output=True, text=True, env=job_env)
print(g.stdout)
if g.stderr.strip():
    print('--- stderr ---'); print(g.stderr[-4000:])
if g.returncode == 2:
    print('COMPUTE_BLOCKED: GPU unavailable / quota denied. No CPU fallback; reconnect a GPU runtime and re-run this cell.')
elif g.returncode != 0:
    raise SystemExit(g.returncode)
else:
    print('STAGE 2 SMOKE COMPLETE ->', f'{PERSISTENT_ROOT}/persistent/reports/stage2_smoke_report.json')


In [ ]:
# Cell H — persistent report + 3D check.
# completed trajectory != accepted binder; pLDDT/ipTM/PAE are not affinity.
import glob, json, os
rep = json.load(open(f'{PERSISTENT_ROOT}/persistent/reports/stage2_smoke_report.json'))
print(json.dumps(rep['flags'], indent=2))
for tag, j in rep.get('jobs', {}).items():
    print(tag, '| status:', j.get('status'),
          '| relaxed:', len(j.get('relaxed_pdb_paths') or []),
          '| accepted-dir count:', j.get('final_design_count'))
try:
    import py3Dmol
except Exception as exc:
    print('py3Dmol unavailable in this kernel:', exc)
else:
    pdbs = sorted(glob.glob(f'{PERSISTENT_ROOT}/*/Trajectory/Relaxed/*.pdb'))
    print('showing', len(pdbs), 'relaxed trajectories')
    for pdb in pdbs:
        v = py3Dmol.view(width=500, height=350)
        v.addModel(open(pdb).read(), 'pdb')
        v.setStyle({'chain': 'A'}, {'cartoon': {'color': '#3c5b6f'}})
        v.setStyle({'chain': 'B'}, {'cartoon': {'color': '#B76E79'}})
        for r in list(range(390, 404)) + list(range(421, 432)):
            v.addStyle({'chain': 'A', 'resi': r}, {'stick': {'color': 'yellow'}})
        v.zoomTo({'chain': 'B'}); v.setTitle(os.path.basename(pdb))
        v.show()


## If a run stops

- **GPU quota / GPU missing**: reconnect a GPU runtime (Runtime → Change
  runtime type → GPU), re-run cells A–G. `COMPUTE_BLOCKED` is not a failure
  of BindCraft.
- **Runtime reset ("/content erased")**: re-run A–G. Env and BindCraft are
  rebuilt; weights restore from Drive cache; completed jobs are skipped.
- Send back `persistent/reports/stage2_smoke_report.json` (+ `.md`) and the
  `persistent/` metadata/log folders. Do not widen sampling beyond this
  notebook without explicit approval.
